# TED 03 — Notebook 01: Definição do problema e preparação para a modelagem
**Pergunta-guia:** que problema de Machine Learning o projeto consegue resolver com os dados preparados na TED 02, e como preparar esses dados sem enganar a avaliação?

Este notebook (1) define o problema, a variável-alvo e as variáveis de entrada, (2) verifica a qualidade do subconjunto `falhas_modelagem.csv`, (3) define a divisão treino/teste e (4) documenta as transformações aplicadas ao texto.

In [1]:
import sys, json, warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, "../../src")
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import modelagem as M, graficos as g
g.aplicar_estilo()
ROOT = Path("../..").resolve()
RJ = ROOT / "reports" / "resultados_modelagem.json"
R = json.loads(RJ.read_text()) if RJ.exists() else {}
def salvar_R(): RJ.write_text(json.dumps(R, ensure_ascii=False, indent=1, default=float))


## 1. Problema de ML, alvo e variáveis
- **Problema:** classificação multiclasse de texto — dado o *sintoma relatado* (texto livre) e o *tipo de aparelho*, prever a **categoria de falha** (taxonomia unificada de 14 categorias da TED 02; 13 aparecem nos dados aptos).
- **Alvo:** `categoria_falha`. **Entradas:** `problema` (texto normalizado) e `categoria` (Celular, Notebook, Tablet, Desktop).
- **Variáveis descartadas e motivo:** `idioma` (é consequência do texto; não estará disponível como rótulo no MVP), `marca` (34% desconhecida e misturada ao aparelho — TED 02, Seção 11.4.2), `status_reparo` (só é conhecido *depois* do reparo — usá-lo seria vazamento de informação) e idade (69% ausente, correlação nula).

In [2]:
df = M.carregar_dados()
print(df.shape)
df[["id_origem", "categoria", "problema", M.COL_TEXTO, "categoria_falha", "ano_evento"]].sample(6, random_state=1)

(4019, 13)


,id_origem,categoria,problema,texto,categoria_falha,ano_evento
3301,rcint_14463,Desktop,start niet op,start niet op,Energia / bateria,2019
2378,restart_15741,Notebook,Windows 10 slow and default programs need to b...,windows slow and default programs need to be set,Software / sistema,2019
3334,rcint_21415,Desktop,aanknop geef rood,aanknop geef rood,Inicialização (boot),2019
366,mobifix-679,Celular,weet niet hoe te updaten - gebrek aan kennis,weet niet hoe te updaten gebrek aan kennis,Software / sistema,2017
577,mobifix-1075,Celular,In einem Gigaset E36 knarzte der Lautsprecher ...,in einem gigaset e knarzte der lautsprecher de...,Áudio / câmera / mídia,2018
893,mobifix-1538,Celular,Ladebuchse kaputt,ladebuchse kaputt,Conectores e portas,2019


In [3]:
dist = df[M.ALVO].value_counts()
R["n_total"] = int(len(df))
R["classes"] = {k: int(v) for k, v in dist.items()}
pd.DataFrame({"registros": dist, "%": (dist / dist.sum() * 100).round(1)})

,registros,%
categoria_falha,,
Software / sistema,804,20.0
Tela,666,16.6
Energia / bateria,622,15.5
Desempenho / superaquecimento,424,10.5
Conectores e portas,320,8.0
Botões / teclado / touchpad,216,5.4
Placa / dano interno,188,4.7
Inicialização (boot),183,4.6
Armazenamento,181,4.5


## 2. Verificações de qualidade antes de modelar

In [4]:
# (a) valores ausentes nas entradas e no alvo
print(df[[M.COL_TEXTO, M.COL_APARELHO, M.ALVO]].isna().sum().to_dict(), "| categoria ausente tratada como 'Desconhecido':", int((df[M.COL_APARELHO] == "Desconhecido").sum()))

# (b) textos idênticos (após normalização) com rótulos diferentes = ruído de rótulo
g_txt = df.groupby(M.COL_TEXTO)[M.ALVO].agg(["size", "nunique"])
rep = g_txt[g_txt["size"] > 1]
conf = rep[rep["nunique"] > 1]
print(f"Textos que se repetem: {len(rep)} ({int(rep['size'].sum())} registros) | com rótulos conflitantes: {len(conf)} ({int(conf['size'].sum())} registros)")
R["textos_repetidos"] = {"textos": int(len(rep)), "registros": int(rep["size"].sum()), "conflitantes": int(len(conf)), "registros_conflitantes": int(conf["size"].sum())}

# (c) tamanho do texto
print(df["n_palavras"].describe().round(1).to_dict())
R["palavras_mediana"] = float(df["n_palavras"].median()); R["pct_ate_3_palavras"] = float((df["n_palavras"] <= 3).mean() * 100)
conf.sort_values("size", ascending=False).head(8).join(df.groupby(M.COL_TEXTO)[M.ALVO].agg(lambda s: " | ".join(sorted(set(s)))).rename("rotulos"))[["size", "rotulos"]]

{'texto': 0, 'categoria': 0, 'categoria_falha': 0} | categoria ausente tratada como 'Desconhecido': 1
Textos que se repetem: 141 (345 registros) | com rótulos conflitantes: 46 (131 registros)
{'count': 4019.0, 'mean': 8.2, 'std': 12.2, 'min': 1.0, '25%': 3.0, '50%': 5.0, '75%': 8.0, 'max': 190.0}


,size,rotulos
texto,,
slow,6,Armazenamento | Desempenho / superaquecimento ...
doesn't charge,5,Conectores e portas | Energia / bateria
power boot,5,Energia / bateria | Inicialização (boot)
not charging,5,Conectores e portas | Energia / bateria
not booting,5,Energia / bateria | Inicialização (boot)
won't turn on,5,Botões / teclado / touchpad | Desempenho / sup...
lecteur,4,Armazenamento | Tela | Áudio / câmera / mídia
not turning on,4,Energia / bateria | Inicialização (boot)


**Leitura.** Textos curtos e repetidos (“won’t turn on”, “slow”) recebem rótulos diferentes de avaliadores diferentes. Isso significa que existe um **teto de desempenho** imposto pelos próprios dados: nenhum modelo consegue acertar um rótulo que humanos não rotularam de forma consistente. Esse ponto é retomado na interpretação dos resultados.

## 3. Estratégia de divisão treino/teste
A TED 02 (Seção 11.3) mostrou que a base reflete o calendário dos eventos (pandemia, mutirões de recondicionamento) e recomendou **divisão que respeite o tempo**. Adotamos:
- **Treino:** eventos até 2018. **Teste:** eventos de 2019 e 2020 (o modelo é avaliado em dados *futuros* em relação ao treino, como será usado no MVP).
- O teste **não é usado** para escolher modelo nem hiperparâmetros: isso é feito por validação cruzada *dentro* do treino (5 dobras estratificadas por rótulo e agrupadas por texto idêntico, para o mesmo texto nunca aparecer em treino e validação).
- Como referência, o notebook 03 compara com uma divisão aleatória estratificada, para medir quanto a deriva temporal pesa.

In [5]:
tr, te = M.dividir_temporal(df)
R["n_treino"], R["n_teste"] = int(len(tr)), int(len(te))
R["pct_teste"] = float(len(te) / len(df) * 100)
R["ano_corte"] = M.ANO_CORTE
print(f"treino = {len(tr)} ({len(tr)/len(df):.1%}) | teste = {len(te)} ({len(te)/len(df):.1%})")
print("classes presentes no teste e ausentes no treino:", set(te[M.ALVO]) - set(tr[M.ALVO]))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.3), gridspec_kw={"width_ratios": [1, 1.3]})
anos = df.groupby("ano_evento").size()
a1.bar(anos.index.astype(str), anos.values, color=[g.SERIES[0] if a <= M.ANO_CORTE else g.SERIES[1] for a in anos.index], width=.7)
a1.set_title("Registros por ano do evento"); a1.set_ylabel("registros"); a1.tick_params(axis="x", labelrotation=60); a1.grid(axis="x", visible=False)
from matplotlib.patches import Patch
a1.legend(handles=[Patch(color=g.SERIES[0], label=f"Treino (até {M.ANO_CORTE})"), Patch(color=g.SERIES[1], label=f"Teste ({M.ANO_CORTE+1}–2020)")], loc="upper left")
d = pd.DataFrame({"Treino": tr[M.ALVO].value_counts(normalize=True) * 100, "Teste": te[M.ALVO].value_counts(normalize=True) * 100}).fillna(0).sort_values("Treino")
y = np.arange(len(d)); a2.barh(y - .2, d["Treino"], .4, color=g.SERIES[0], label="Treino"); a2.barh(y + .2, d["Teste"], .4, color=g.SERIES[1], label="Teste")
a2.set_yticks(y); a2.set_yticklabels(d.index, fontsize=8.5); a2.grid(axis="y", visible=False); a2.set_xlabel("% dos registros do conjunto"); a2.set_title("Distribuição das classes"); a2.legend(loc="lower right")
fig.tight_layout(); g.salvar(fig, "fig13_divisao_dados"); plt.show()
R["dist_treino"] = {k: int(v) for k, v in tr[M.ALVO].value_counts().items()}; R["dist_teste"] = {k: int(v) for k, v in te[M.ALVO].value_counts().items()}

treino = 2224 (55.3%) | teste = 1795 (44.7%)
classes presentes no teste e ausentes no treino: set()


In [6]:
# sobreposição de textos entre treino e teste (se for alta, o teste mede "memória", não generalização)
vistos = set(tr[M.COL_TEXTO])
te["texto_visto"] = te[M.COL_TEXTO].isin(vistos)
R["pct_teste_texto_visto"] = float(te["texto_visto"].mean() * 100)
print(f"{te['texto_visto'].mean():.1%} dos textos de teste aparecem (idênticos) no treino")
# guarda a divisão para reprodução
div = pd.concat([tr[["id_origem"]].assign(conjunto="treino"), te[["id_origem"]].assign(conjunto="teste")])
div.to_csv(ROOT / "data" / "processed" / "divisao_treino_teste_ted03.csv", index=False)
salvar_R()

5.0% dos textos de teste aparecem (idênticos) no treino


## 4. Transformações aplicadas (resumo)
| Etapa | O que foi feito | Por quê |
|---|---|---|
| Normalização do texto | minúsculas, remoção de URLs, números e pontuação; acentos preservados | reduz variações sem perder palavras de línguas diferentes |
| Vetorização | TF-IDF de **palavras** (1–2 gramas) e/ou de **caracteres** (2–5 gramas) | converte texto em números; n-gramas de caracteres toleram erros de digitação, flexões e palavras compostas do alemão/holandês |
| Aparelho | codificação *one-hot* | variável categórica nominal, sem ordem |
| Normalização numérica | não aplicada | não há variável numérica nas entradas (idade foi descartada) |
| Desbalanceamento | pesos de classe balanceados (`class_weight="balanced"`) | “Dano por líquido” tem 7 exemplos no treino; sem pesos o modelo ignoraria as classes raras |
Todas as transformações ficam **dentro** do *pipeline* e são ajustadas só com o treino de cada dobra, evitando vazamento do teste para o treino.